# Gold: arrears and vacancy star schema

The layer a semantic model can sit on directly. Conformed dimensions, additive facts,
one row per grain, no surprises.

| | |
| --- | --- |
| **Reads** | `silver_*` conformed tables |
| **Writes** | four dimensions and three facts |

### How arrears is actually calculated

Arrears is not a column in any source system — it is the residue of charges that
receipts have not covered. Receipts are allocated **oldest charge first**, which is how
a rent account is actually settled, and the age of the oldest still-unpaid charge
determines the aging bucket.

The alternative — comparing this month's charge to this month's payment — is easier and
wrong: a household that pays late every month looks permanently in arrears, and one
that misses a month then catches up looks fine when it never cleared the original debt.

In [ ]:
PIPELINE_RUN_ID = ""

In [ ]:
import json
from datetime import datetime, timezone

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, DateType, DecimalType, DoubleType, IntegerType, StringType,
    StructField, StructType,
)

_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_LAKEHOUSE_ID = {}


def lake_table(lakehouse, table_name):
    if lakehouse not in _LAKEHOUSE_ID:
        _LAKEHOUSE_ID[lakehouse] = notebookutils.lakehouse.get(
            lakehouse, workspaceId=_WS).id
    return spark.read.format("delta").load(
        f"abfss://{_WS}@{_ONELAKE}/{_LAKEHOUSE_ID[lakehouse]}/Tables/{table_name}")


def silver(name):
    return lake_table("silver_lakehouse", name)


RUN_ID = PIPELINE_RUN_ID or (silver("silver_dim_unit")
                             .orderBy(F.desc("generated_at_utc"))
                             .select("run_id").first()["run_id"])
GENERATED_AT = datetime.now(timezone.utc).isoformat()
print("run_id:", RUN_ID)


def write_run_scoped(frame, table_name):
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        (frame.write.format("delta").mode("overwrite")
            .partitionBy("run_id").saveAsTable(table_name))
    except Exception as error:
        print(f"  WARNING {table_name} schema changed - replacing all runs "
              f"({str(error).splitlines()[0][:100]})")
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
        try:
            (frame.write.format("delta").mode("overwrite").partitionBy("run_id")
                .option("overwriteSchema", "true").saveAsTable(table_name))
        finally:
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    print(f"  {table_name:32} {frame.count():>9,} rows")

## 1. Dimensions

In [ ]:
buildings = silver("silver_dim_building").filter(F.col("run_id") == RUN_ID)
units = silver("silver_dim_unit").filter(F.col("run_id") == RUN_ID)
households = silver("silver_dim_household").filter(F.col("run_id") == RUN_ID)
charges = silver("silver_fact_rent_charge").filter(F.col("run_id") == RUN_ID)
receipts = silver("silver_fact_receipt").filter(F.col("run_id") == RUN_ID)
occupancy = silver("silver_fact_occupancy").filter(F.col("run_id") == RUN_ID)
turnaround = silver("silver_fact_unit_turnaround").filter(F.col("run_id") == RUN_ID)

dim_building = buildings.select(
    "building_id", "building_name", "ward_name", "region", "property_type",
    "year_built", "total_units", "run_id",
    F.lit(GENERATED_AT).alias("generated_at_utc"))

dim_unit = (units.join(buildings.select("building_id", "building_name", "ward_name",
                                        "region", "property_type"),
                       "building_id", "left")
            .select("unit_id", "building_id", "building_name", "ward_name", "region",
                    "property_type", "unit_number", "bedroom_count",
                    F.when(F.col("bedroom_count") == 0, F.lit("Bachelor"))
                     .when(F.col("bedroom_count") == 1, F.lit("1 bedroom"))
                     .when(F.col("bedroom_count") == 2, F.lit("2 bedroom"))
                     .when(F.col("bedroom_count") == 3, F.lit("3 bedroom"))
                     .otherwise(F.lit("4+ bedroom")).alias("unit_size"),
                    "tenure_type", "is_accessible", "run_id",
                    F.lit(GENERATED_AT).alias("generated_at_utc")))

dim_household = households.select(
    F.col("household_ref").alias("household_key"), "unit_id", "household_size",
    "income_band", F.col("subsidy_type").alias("tenure_type"),
    "move_in_date", "move_out_date",
    F.when(F.col("move_out_date").isNull(), F.lit("Active"))
     .otherwise(F.lit("Former")).alias("tenancy_status"),
    "run_id", F.lit(GENERATED_AT).alias("generated_at_utc"))

# A real date dimension, built from the charge window rather than hardcoded.
bounds = charges.agg(F.min("period_start").alias("lo"),
                     F.max("period_start").alias("hi")).first()
dates = (spark.sql(f"SELECT explode(sequence(to_date('{bounds['lo']}'), "
                   f"last_day(to_date('{bounds['hi']}')), interval 1 day)) AS date")
         .select(
             F.col("date"),
             F.date_format("date", "yyyyMMdd").cast(IntegerType()).alias("date_key"),
             F.year("date").alias("year"),
             F.quarter("date").alias("quarter"),
             F.month("date").alias("month_number"),
             F.date_format("date", "MMM yyyy").alias("month_name"),
             # month_name is text and would sort alphabetically. sortByColumn
             # needs a 1:1 partner, so this is month-grained, not daily.
             F.date_format("date", "yyyyMM").cast(IntegerType()).alias("year_month"),
             F.trunc("date", "month").alias("period_start"),
             F.last_day("date").alias("period_end"),
             F.concat(F.lit("FY"), F.when(F.month("date") >= 4, F.year("date") + 1)
                      .otherwise(F.year("date"))).alias("fiscal_year"),
             F.lit(RUN_ID).alias("run_id")))
dim_date = dates

print("dimensions built")

## 2. Arrears: allocate receipts oldest-charge-first

Done in pandas because the allocation is inherently sequential per household. At this
volume that is the pragmatic choice; at production volume it becomes a windowed Spark
job or an incremental merge on the prior month's closing balance.

In [ ]:
charge_pd = (charges.select("household_key", "unit_id", "period_start", "charge_amount")
             .toPandas())
receipt_pd = (receipts.select("household_key", "posting_date", "payment_amount")
              .toPandas())

charge_pd["charge_amount"] = charge_pd["charge_amount"].astype(float)
receipt_pd["payment_amount"] = receipt_pd["payment_amount"].astype(float)
charge_pd["period_start"] = pd.to_datetime(charge_pd["period_start"])
receipt_pd["posting_date"] = pd.to_datetime(receipt_pd["posting_date"])
receipt_pd["period_start"] = receipt_pd["posting_date"].values.astype("datetime64[M]")

periods = sorted(charge_pd["period_start"].unique())
charges_by_household = {k: g.sort_values("period_start")
                        for k, g in charge_pd.groupby("household_key")}
receipts_by_household = {k: g.groupby("period_start")["payment_amount"].sum().to_dict()
                         for k, g in receipt_pd.groupby("household_key")}
unit_of_household = charge_pd.groupby("household_key")["unit_id"].first().to_dict()

BUCKETS = [(0, "Current"), (1, "1-30 days"), (2, "31-60 days"),
           (3, "61-90 days"), (4, "Over 90 days")]

snapshot_rows = []
for household, frame in charges_by_household.items():
    receipts_for = receipts_by_household.get(household, {})
    # open[i] = amount still owed against the charge raised in periods[i]
    open_charges = []
    charge_lookup = dict(zip(frame["period_start"], frame["charge_amount"]))

    for period in periods:
        if period in charge_lookup:
            open_charges.append([period, float(charge_lookup[period])])

        # Settle oldest first.
        cash = float(receipts_for.get(period, 0.0))
        while cash > 0.005 and open_charges:
            oldest = open_charges[0]
            applied = min(cash, oldest[1])
            oldest[1] -= applied
            cash -= applied
            if oldest[1] <= 0.005:
                open_charges.pop(0)

        balance = round(sum(item[1] for item in open_charges), 2)
        if not charge_lookup.get(period) and balance <= 0.005:
            continue

        # Age on the oldest charge still carrying a balance.
        if open_charges:
            oldest_period = open_charges[0][0]
            months_back = ((period.year - oldest_period.year) * 12
                           + period.month - oldest_period.month)
        else:
            months_back = 0
        bucket = BUCKETS[min(months_back, 4)][1]

        snapshot_rows.append({
            "run_id": RUN_ID,
            "household_key": household,
            "unit_id": unit_of_household.get(household),
            "period_start": period.date(),
            "charge_raised": round(float(charge_lookup.get(period, 0.0)), 2),
            "receipts_applied": round(float(receipts_for.get(period, 0.0)) - cash, 2),
            "closing_balance": balance,
            "arrears_bucket": bucket if balance > 0.005 else "Current",
            "months_in_arrears": int(months_back if balance > 0.005 else 0),
            "is_in_arrears": bool(balance > 0.005),
        })

print(f"arrears snapshots: {len(snapshot_rows):,}")

In [ ]:
# Build with DoubleType -- Spark refuses to accept a Python float into DecimalType when
# constructing from rows -- then cast to decimal inside the DataFrame, where the
# conversion is exact.
SNAPSHOT_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("household_key", StringType(), True),
    StructField("unit_id", StringType(), True),
    StructField("period_start", DateType(), True),
    StructField("charge_raised", DoubleType(), True),
    StructField("receipts_applied", DoubleType(), True),
    StructField("closing_balance", DoubleType(), True),
    StructField("arrears_bucket", StringType(), True),
    StructField("months_in_arrears", IntegerType(), True),
    StructField("is_in_arrears", BooleanType(), True),
])
def coerce(value, spark_type):
    """createDataFrame does no type coercion -- an int where the schema says double is a
    hard failure. sum([]) returns int 0, which is exactly how this bites."""
    if value is None:
        return None
    if isinstance(spark_type, DoubleType):
        return float(value)
    if isinstance(spark_type, IntegerType):
        return int(value)
    if isinstance(spark_type, BooleanType):
        return bool(value)
    if isinstance(spark_type, StringType):
        return str(value)
    return value


fields = list(SNAPSHOT_SCHEMA.fields)
fact_arrears = spark.createDataFrame(
    [tuple(coerce(row.get(field.name), field.dataType) for field in fields)
     for row in snapshot_rows],
    schema=SNAPSHOT_SCHEMA)
for _money in ("charge_raised", "receipts_applied", "closing_balance"):
    fact_arrears = fact_arrears.withColumn(
        _money, F.col(_money).cast(DecimalType(12, 2)))

# Denormalise the grain the report filters on, so the model needs no bridge tables.
fact_arrears = (fact_arrears
                .join(dim_unit.select("unit_id", "building_id", "ward_name", "region",
                                      "tenure_type", "unit_size"), "unit_id", "left")
                .withColumn("date_key",
                            F.date_format("period_start", "yyyyMMdd").cast(IntegerType())))

## 3. Vacancy and turnaround

In [ ]:
# A unit is vacant in a month if no occupancy spell covers that month's start.
month_grid = (dim_date.select("period_start").distinct()
              .crossJoin(dim_unit.select("unit_id", "building_id", "ward_name", "region",
                                         "tenure_type", "unit_size")))

covered = (month_grid.join(
    occupancy.select("unit_id", "occupied_from", "occupied_to"), "unit_id", "left")
    .withColumn("is_covered",
                (F.col("occupied_from") <= F.col("period_start"))
                & (F.col("occupied_to").isNull()
                   | (F.col("occupied_to") >= F.col("period_start"))))
    .groupBy("period_start", "unit_id", "building_id", "ward_name", "region",
             "tenure_type", "unit_size")
    .agg(F.max(F.when(F.col("is_covered"), F.lit(1)).otherwise(F.lit(0)))
         .alias("occupied_flag")))

# Expected rent for a vacant unit, used to quantify revenue forgone.
average_rent = (charges.groupBy("unit_id")
                .agg(F.avg("charge_amount").cast(DecimalType(12, 2)).alias("avg_rent")))

fact_vacancy = (covered.join(average_rent, "unit_id", "left")
                .select(
                    F.lit(RUN_ID).alias("run_id"),
                    "unit_id", "building_id", "ward_name", "region", "tenure_type",
                    "unit_size", "period_start",
                    F.date_format("period_start", "yyyyMMdd").cast(IntegerType())
                     .alias("date_key"),
                    F.col("occupied_flag").cast(IntegerType()).alias("occupied_flag"),
                    (F.lit(1) - F.col("occupied_flag")).cast(IntegerType())
                     .alias("vacant_flag"),
                    F.when(F.col("occupied_flag") == 0, F.coalesce("avg_rent", F.lit(0)))
                     .otherwise(F.lit(0)).cast(DecimalType(12, 2))
                     .alias("revenue_forgone")))

fact_turnaround = (turnaround
                   .join(dim_unit.select("unit_id", "building_id", "ward_name", "region",
                                         "tenure_type", "unit_size"), "unit_id", "left")
                   .select(
                       "run_id", "work_order_id", "unit_id", "building_id", "ward_name",
                       "region", "tenure_type", "unit_size", "vacated_date",
                       "ready_to_rent_date", "turnaround_category", "turnaround_days",
                       F.date_format("vacated_date", "yyyyMMdd").cast(IntegerType())
                        .alias("date_key"),
                       F.col("turnaround_days").isNotNull().alias("is_complete")))

In [ ]:
print("writing gold:")
write_run_scoped(dim_date, "gold_dim_date")
write_run_scoped(dim_building, "gold_dim_building")
write_run_scoped(dim_unit, "gold_dim_unit")
write_run_scoped(dim_household, "gold_dim_household")
write_run_scoped(fact_arrears, "gold_fact_arrears_snapshot")
write_run_scoped(fact_vacancy, "gold_fact_unit_month")
write_run_scoped(fact_turnaround, "gold_fact_turnaround")

In [ ]:
latest = fact_arrears.agg(F.max("period_start")).first()[0]
summary = (fact_arrears.filter(F.col("period_start") == latest)
           .agg(F.sum("closing_balance").alias("total_arrears"),
                F.sum(F.when(F.col("is_in_arrears"), 1).otherwise(0)).alias("households_in_arrears"),
                F.count("*").alias("households")).first())
vacancy_now = (fact_vacancy.filter(F.col("period_start") == latest)
               .agg(F.sum("vacant_flag").alias("vacant"),
                    F.count("*").alias("units"),
                    F.sum("revenue_forgone").alias("forgone")).first())

print(json.dumps({
    "period": str(latest),
    "total_arrears": float(summary["total_arrears"] or 0),
    "households_in_arrears": summary["households_in_arrears"],
    "households": summary["households"],
    "arrears_rate_pct": round(100 * summary["households_in_arrears"] / summary["households"], 2),
    "units_vacant": vacancy_now["vacant"],
    "units": vacancy_now["units"],
    "vacancy_rate_pct": round(100 * vacancy_now["vacant"] / vacancy_now["units"], 2),
    "monthly_revenue_forgone": float(vacancy_now["forgone"] or 0),
}, indent=1))

In [ ]:
display(spark.read.table("gold_fact_arrears_snapshot")
        .filter((F.col("run_id") == RUN_ID) & (F.col("period_start") == latest))
        .groupBy("arrears_bucket")
        .agg(F.count("*").alias("households"),
             F.sum("closing_balance").alias("balance"))
        .orderBy("arrears_bucket"))